# Trabalho Prático: Processamento de Linguagem Natural (PLN)
## Classificação de Sentimentos com TF-IDF e Regressão Logística

**Objetivo Geral:** O objetivo deste projeto é construir e avaliar um modelo de Machine Learning capaz de classificar frases de feedback de alunos em sentimentos **Positivos (1)** ou **Negativos (0)**.

**Etapas do Projeto:**
1. Expansão do Dataset para 100 frases balanceadas.
2. Análise exploratória dos dados textuais.
3. Engenharia de recursos (Extração de características com TF-IDF).
4. Divisão dos dados e treinamento do modelo preditivo.
5. Avaliação do desempenho do modelo através de métricas de acurácia.
6. Testes práticos com frases totalmente inéditas.

In [66]:
# Importando as bibliotecas essenciais para manipulação de dados e Machine Learning
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


## 1. Expansão do Dataset (100 Frases)
**Objetivo:** Coletar e expandir a base de dados original para um total de 100 observações. Para evitar problemas de viés (onde o modelo aprende a chutar apenas a classe majoritária), o dataset foi construído de forma perfeitamente balanceada: **50 frases positivas** e **50 frases negativas**.


In [67]:
import random

# Definimos uma "semente" para que o sorteio aleatório seja sempre o mesmo
# Toda vez que você rodar, ele vai gerar as exatas mesmas 600 frases
random.seed(42)

# 1. Vocabulário base (Peças de Lego)
sujeitos = ["A aula", "O curso", "O professor", "A explicação", "O material", "A didática", "A plataforma", "O conteúdo", "O exercício", "A dinâmica", "O suporte", "A avaliação", "A monitoria", "O projeto", "O tema"]
verbos = ["foi", "está", "é", "me pareceu", "se mostrou", "sempre é", "tem sido"]

# Palavras de sentimento POSITIVO
adj_pos = ["excelente", "incrível", "muito bom", "fantástico", "esclarecedor", "maravilhoso", "perfeito", "ótimo", "muito produtivo", "sensacional", "inspirador", "útil", "muito claro", "bem organizado", "surpreendente", "impecável", "nota 10", "acima da média", "rico", "enriquecedor", "didático", "dinâmico", "direto ao ponto", "fascinante"]
comp_pos = ["!", ".", " hoje.", " de verdade.", ", parabéns!", " mesmo.", ", gostei muito.", ", super recomendo.", ", aprendi bastante.", " para minha formação."]

# Palavras de sentimento NEGATIVO
adj_neg = ["péssimo", "horrível", "muito ruim", "confuso", "desorganizado", "chato", "entediante", "difícil de entender", "inútil", "frustrante", "decepcionante", "complicado demais", "mal preparado", "lento", "superficial", "cansativo", "tedioso", "fraco", "desatualizado", "irritante", "mal estruturado", "muito denso", "complicado", "vago"]
comp_neg = ["!", ".", " hoje.", " infelizmente.", ", não gostei.", " mesmo.", ", não recomendo.", ", não aprendi nada.", " para mim.", ", me deixou confuso."]

# 2. Gerando as frases de forma combinatória e aleatória usando "sets" para não ter frases repetidas
frases_positivas = set()
while len(frases_positivas) < 300:
    frase = f"{random.choice(sujeitos)} {random.choice(verbos)} {random.choice(adj_pos)}{random.choice(comp_pos)}"
    frases_positivas.add(frase)

frases_negativas = set()
while len(frases_negativas) < 300:
    frase = f"{random.choice(sujeitos)} {random.choice(verbos)} {random.choice(adj_neg)}{random.choice(comp_neg)}"
    frases_negativas.add(frase)

# 3. Juntando tudo na lista final
frases = list(frases_positivas) + list(frases_negativas)
rotulos = [1]*300 + [0]*300

print(f"Dataset expandido com sucesso usando Geração Aleatória!")
print(f"Total de frases: {len(frases)}")
print(f"Positivas (1): {rotulos.count(1)} | Negativas (0): {rotulos.count(0)}")

Dataset expandido com sucesso usando Geração Aleatória!
Total de frases: 600
Positivas (1): 300 | Negativas (0): 300


## 2. Exploração e Análise de Dados Textuais
**Objetivo:** Compreender a estrutura básica do dataset criado. Aqui validamos programaticamente se a quantidade de exemplos bate com o planejado e exibimos amostras aleatórias de dados reais de cada classe para conferência visual.

In [68]:
# Convertendo temporariamente em um DataFrame do Pandas para análise ágil
df_exploracao = pd.DataFrame({'Texto': frases, 'Sentimento': rotulos})

# 1. Contagem de frases por classe
contagem = df_exploracao['Sentimento'].value_counts()
print("=== DISTRIBUIÇÃO DAS CLASSES ===")
print(f"Frases Positivas (Classe 1): {contagem[1]}")
print(f"Frases Negativas (Classe 0): {contagem[0]}\n")

# 2. Exibição de exemplos de frases de cada tipo
print("=== AMOSTRA DE FRASES POSITIVAS ===")
print(df_exploracao[df_exploracao['Sentimento'] == 1]['Texto'].head(3).to_string(index=False))

print("\n=== AMOSTRA DE FRASES NEGATIVAS ===")
print(df_exploracao[df_exploracao['Sentimento'] == 0]['Texto'].head(3).to_string(index=False))

=== DISTRIBUIÇÃO DAS CLASSES ===
Frases Positivas (Classe 1): 300
Frases Negativas (Classe 0): 300

=== AMOSTRA DE FRASES POSITIVAS ===
       A monitoria tem sido fantástico de verdade.
A avaliação sempre é direto ao ponto, aprendi b...
                 O curso foi direto ao ponto hoje.

=== AMOSTRA DE FRASES NEGATIVAS ===
                   A monitoria está desorganizado!
                   A aula está inútil, não gostei.
O professor se mostrou complicado demais, não r...


## 3. Engenharia de Recursos: Engenharia do TF-IDF
**Objetivo:** Algoritmos matemáticos não processam textos em sua forma natural. Precisamos mapear palavras para números. O método **TF-IDF (Term Frequency-Inverse Document Frequency)** calcula a relevância de cada palavra ponderando quantas vezes ela aparece em uma frase específica em relação à sua raridade ao longo de todo o corpus de 100 frases.

Abaixo, transformamos o texto e usamos um DataFrame para visualizar uma amostra da tabela numérica resultante.

In [69]:
# Modificação: Adicionamos o parâmetro ngram_range=(1, 2)
# Isso faz o modelo analisar palavras isoladas E combinações de duas palavras (ex: "não gostei", "muito bom")
vetorizador = TfidfVectorizer(ngram_range=(1, 2))

# Mapeando o vocabulário e convertendo as frases em vetores numéricos
X = vetorizador.fit_transform(frases)
y = rotulos

# Criando um DataFrame para inspecionar visualmente a nova matriz (agora com bigramas)
df_tfidf = pd.DataFrame(X.toarray(), columns=vetorizador.get_feature_names_out())

print(f"Formato da Matriz (Agora com mais colunas devido aos Bigramas): {df_tfidf.shape}")
df_tfidf.head(5)

Formato da Matriz (Agora com mais colunas devido aos Bigramas): (600, 755)


,10,10 aprendi,10 de,10 gostei,10 mesmo,10 para,10 parabéns,10 super,acima,acima da,...,ótimo super,útil,útil aprendi,útil de,útil gostei,útil hoje,útil mesmo,útil para,útil parabéns,útil super
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Divisão do Dataset e Treinamento do Modelo
**Objetivo:** Dividir os dados transformados em duas partes: **Treinamento (75%)** para o algoritmo aprender os padrões textuais, e **Teste (25%)** para simular como o modelo se comporta com dados não vistos. Após a divisão, instanciamos o algoritmo de **Regressão Logística** e o ajustamos com o método `.fit()`.

In [70]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Criamos uma lista de palavras neutras (stop words) para limpar o texto
stop_words_pt = ['eu', 'o', 'a', 'os', 'as', 'de', 'do', 'da', 'em', 'um', 'uma', 'que', 'com', 'no', 'na', 'para', 'foi', 'será', 'uma', 'pela', 'pelo']

# Configuração focada em alta acurácia para datasets pequenos:
# min_df=2 faz o modelo ignorar palavras raras que só aparecem 1 vez, evitando que ele decore o treino
vetorizador = TfidfVectorizer(
    ngram_range=(1, 1),
    stop_words=stop_words_pt,
    min_df=2,
    lowercase=True
)

# Mapeando o vocabulário e convertendo as frases em números
X = vetorizador.fit_transform(frases)
y = rotulos

# Criando o DataFrame para visualização
df_tfidf = pd.DataFrame(X.toarray(), columns=vetorizador.get_feature_names_out())

print(f"Formato da Matriz filtrada: {df_tfidf.shape}")
print("Texto limpo e pronto para o treinamento!")
df_tfidf.head(5)

Formato da Matriz filtrada: (600, 95)
Texto limpo e pronto para o treinamento!


,10,acima,ao,aprendi,aula,avaliação,bastante,bem,bom,cansativo,...,superficial,suporte,surpreendente,tedioso,tem,tema,vago,verdade,ótimo,útil
0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.344214,0.0,0.0,0.459785,0.00000,0.0
1,0.0,0.0,0.446744,0.295193,0.0,0.33198,0.366424,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.00000,0.0
2,0.0,0.0,0.501302,0.000000,0.0,0.00000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.00000,0.0
3,0.0,0.0,0.000000,0.392481,0.0,0.00000,0.487189,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.55399,0.0
4,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.00000,0.0


## 5. Avaliação da Acurácia do Modelo
**Objetivo:** Testar estatisticamente a eficácia do classificador. O modelo realiza previsões em cima do conjunto `X_teste` e a função `accuracy_score` mede a porcentagem de acertos totais comparando o resultado previsto com os gabaritos reais (`y_teste`).

In [71]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Separando dados para treino (75%) e teste (25%)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

# Modelo configurado com peso equilibrado (C=1.0) para evitar o vício
modelo = LogisticRegression(C=1.0, max_iter=1000)

# Treinando o modelo
modelo.fit(X_treino, y_treino)

# Calculando a nova acurácia no conjunto de teste
previsoes_teste = modelo.predict(X_teste)
acuracia_final = accuracy_score(y_teste, previsoes_teste) * 100

print("=" * 40)
print(f" NOVA ACURÁCIA DO MODELO: {acuracia_final:.2f}%")
print("=" * 40)

 NOVA ACURÁCIA DO MODELO: 100.00%


## 6. Testes Práticos e Tabela Comparativa (Resultados Esperados vs. Obtidos)
**Objetivo:** Simular o modelo operando em produção. Criamos um input para frases inteiramente novas que **não pertencem** ao dataset original. Passamos essas frases pelo pipeline (vetorização TF-IDF do modelo), geramos a previsão e organizamos tudo em uma tabela final clara contendo: a frase, o sentimento esperado, o sentimento previsto e se houve acerto.

In [72]:
# 1. Inicializando as listas para guardar o que o usuário digitar
novas_frases = []
sentimentos_esperados = []

print("=== MÓDULO DE TESTES INTERATIVOS ===")

print("Digite as frases inéditas para testar o modelo.")
print("Quando terminar de inserir as frases, digite 'sair' no campo de texto para gerar a tabela.\n")

while True:
    # Captura a frase do usuário
    frase_usuario = input("Digite uma frase inédita (ou 'sair'): ").strip()

    # Condição de parada do loop
    if frase_usuario.lower() == 'sair':
        break

    if not frase_usuario:
        print("A frase não pode ser vazia. Tente novamente.\n")
        continue

    # Captura o sentimento esperado para poder avaliar se o modelo acertou
    print("Qual é o sentimento REAL/ESPERADO para esta frase?")
    print(" 1 - Positivo")
    print(" 0 - Negativo")

    try:
        esperado = int(input("Escolha o rótulo (1 ou 0): "))
        if esperado not in [0, 1]:
            print("Opção inválida! Esta frase foi descartada. Tente novamente.\n")
            continue
    except ValueError:
        print("Entrada inválida! Digite apenas o número 1 ou 0. Esta frase foi descartada.\n")
        continue

    # Armazena os dados validados nas listas
    novas_frases.append(frase_usuario)
    sentimentos_esperados.append(esperado)
    print("-" * 50)

# 2. Processamento e geração da tabela após o usuário digitar 'sair'
if len(novas_frases) > 0:
    # Transformando as frases digitadas usando o MESMO vetorizador do modelo
    X_novas = vetorizador.transform(novas_frases)

    # Executando as previsões do modelo de Regressão Logística
    previsoes_novas = modelo.predict(X_novas)

    # Dicionário de mapeamento para deixar a tabela visualmente clara
    mapa_rotulo = {1: "Positivo", 0: "Negativo"}

    # 3. Montando a tabela comparativa final exigida pelo professor
    df_comparacao = pd.DataFrame({
        'Frase': novas_frases,
        'Sentimento Esperado': [mapa_rotulo[s] for s in sentimentos_esperados],
        'Sentimento Previsto': [mapa_rotulo[p] for p in previsoes_novas],
        'Resultado (Acertou?)': ['Sim' if s == p else 'Não' for s, p in zip(sentimentos_esperados, previsoes_novas)]
    })

    print("\n" + "="*25 + " TABELA COMPARATIVA DE RESULTADOS " + "="*25)
    # Exibe a tabela formatada no Jupyter
    display(df_comparacao)
else:
    print("\nNenhuma frase foi inserida para a realização dos testes.")

=== MÓDULO DE TESTES INTERATIVOS ===
Digite as frases inéditas para testar o modelo.
Quando terminar de inserir as frases, digite 'sair' no campo de texto para gerar a tabela.

Digite uma frase inédita (ou 'sair'): Que tal falar mais alto para os vizinhos também participarem da nossa conversa?
Qual é o sentimento REAL/ESPERADO para esta frase?
 1 - Positivo
 0 - Negativo
Escolha o rótulo (1 ou 0): 0
--------------------------------------------------
Digite uma frase inédita (ou 'sair'): sair

========================= TABELA COMPARATIVA DE RESULTADOS =========================


,Frase,Sentimento Esperado,Sentimento Previsto,Resultado (Acertou?)
0,Que tal falar mais alto para os vizinhos també...,Negativo,Positivo,Não
